---
## Standardize Jira ticket descriptions with LLM
- Jira tickets descriptions vary and thus the ticket consumer needs to navigate ambiguity. 
- If the descriptions were in a standardized format and explained in a simple manner, and in detail, this would improve project management.
- The descriptions of tickets are processed by OpenAI gtp-4o-mini LLM.
- This notebooks does not perform evaluation or fine-tuning.

### 📅 Date  
`2025-04-30`

### 📝 Author  
**Samuli**

### 🎯 Objective  
- **How does it work?**
1. Install client and import dependencies. 
2. Load JSONL file from "Files" read it into spark dataframe and then flatten the df and then convert it back to JSONL. (OpenAI rest API only reads JSONL)
3. Get Azure token and items from Azure Key Vault. The OpenAI API key is stored in the key vault and this is what we want to fetch. 
4. Define endpoint, model and model version etc.
5. Write instruction(prompt) for the AI model.
6. Limit tokens per minute and send payload to API in batches. Loop the batches and append the response results to a list.
7. Create dataframe from the list and join the new data to the original dataframe.
8. Write the dataframe to delta table.
- **What is the expected outcome?**
    The response is the LLM output text that matches the instructions for each Jira ticket description.
    The final product is a Data Frame that is loaded to LH_bronze as delta table called 'system_of_work_output'.
---


### Install client and import dependecies

In [ ]:
%pip install -U openai

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType
from pyspark.sql.functions import  col, when, to_date, to_timestamp, date_format, row_number
from pyspark.sql.window import Window
from notebookutils import mssparkutils
import time
from pyspark.sql import SparkSession
import jwt
import json
from openai import AzureOpenAI, RateLimitError

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 9, Finished, Available, Finished)

In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("system_of_work_agent").getOrCreate()

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 10, Finished, Available, Finished)

In [25]:
source_path = "abfss://9de4aec6-b8c0-4cc6-aa61-5895dd146525@onelake.dfs.fabric.microsoft.com/bac0d223-cc50-4b9c-98f4-16da4e2960a7/Files/system-of-work/hiq-reporting-issues.json"
tableName = 'system_of_work_output'

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 32, Finished, Available, Finished)

### Import the source file
Load JSONL file from "Files" read it into spark dataframe and then flatten the df and then convert it back to JSONL. (OpenAI rest API only reads JSONL)

In [5]:
# Read JSON file
df = spark.read.json(source_path)
issues = df.select("issue.*")
# Flatten JSON
flat_issues = issues.select(
    col("title"),
    col("description"),
    col("assignee"),
    col("comments"),
    col("other_details.priority").alias("priority"),
    col("other_details.status").alias("status"),
    col("other_details.created").alias("created"),
    col("other_details.updated").alias("updated")
)

# Define a window
window_spec = Window.orderBy("title")  

# Add the auto-incrementing column
flat_issues_id = flat_issues.withColumn("id", row_number().over(window_spec))
flat_issues = flat_issues_id.limit(50)

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 12, Finished, Available, Finished)

In [9]:
# Convert to dict and call it input_data 
input_data = flat_issues.toJSON().map(lambda x: json.loads(x)).collect()

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 16, Finished, Available, Finished)

### Define Azure tools
Get Azure token and items from Azure Key Vault. The OpenAI API key is stored in the key vault and this is what we want to fetch.

In [10]:
# check if running in local environment or fabric capacity
def is_fabric():
    import os
    return os.path.exists('/synfs')

# get azure token --
def get_azure_token():
    try:
        if not is_fabric():

            # type: 'az login' in CLI. 
            from azure.identity import DefaultAzureCredential
            from datetime import datetime, timedelta
            import time

            # Get authentication token using Azure Identity CLI
            credential = DefaultAzureCredential()
            token = credential.get_token("https://storage.azure.com/.default")

            print("Running in local environment. Authentication successful ✅")

            if hasattr(token, 'expires_on'):
                current_time = time.time()
                expiry_time = token.expires_on
                
                # Calculate remaining time
                remaining_seconds = expiry_time - current_time
                remaining_time = timedelta(seconds=remaining_seconds)

                expiry_datetime = datetime.fromtimestamp(expiry_time)
                
                print(f"Token expires on: {expiry_datetime}")
                print(f"Token valid for: {remaining_time}")

    except Exception as e:
                print(f"Error in defining run environment: {e}")
                raise e
    return token 

def get_azure_kv_token(secret_name):
    if secret_name:
        if not is_fabric():
            # Using Azure Identity library
            from azure.keyvault.secrets import SecretClient
            from azure.identity import DefaultAzureCredential
            credential = DefaultAzureCredential()
            key_vault_url = 'https://hiq-reporting-kv.vault.azure.net/'
            kv_client = SecretClient(vault_url=key_vault_url, credential=credential)
            token = kv_client.get_secret(secret_name).value
        else:
            import notebookutils
            key_vault_url = 'https://hiq-reporting-kv.vault.azure.net/'
            token = notebookutils.credentials.getSecret(f'{key_vault_url}',secret_name)
    else:
         print('Secret name is empty!')
         return False
    return token

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 17, Finished, Available, Finished)

### Prepare variables for client
Get Azure token and items from Azure Key Vault. The OpenAI API key is stored in the key vault and this is what we want to fetch.

In [11]:
# setup api key and create client variable

model_version= "2024-12-01-preview"
model_name = "gpt-4o-mini"
deployment = "gpt-4o-mini"
endpoint = 'https://hiqreportingai5144645464.openai.azure.com/'
key_name_in_kv = 'gpt-4o-mini-api-key'
openai_api_key = get_azure_kv_token(key_name_in_kv)

client = AzureOpenAI(api_key=openai_api_key, api_version=model_version, azure_endpoint=endpoint)

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 18, Finished, Available, Finished)

### Prompting
Write instructions(prompt) for the AI model.

In [12]:
instruction = """
You are a helpful assistant that analyzes Jira tickets.
Standardize the following Jira ticket descriptions and format the descriptions to be more informative and cover topics "Miksi tehdään?" and "Tehtävät". 
The "Miksi tehdään?" should answer to question "why?", why the task is important and to whom are we making it. "Miksi tehdään?" is the story section of the issue.
The "Tehtävät" should highlight what should be done in order to close the ticket.
The "Tehtävät" section can have tasks in bullet points if the task has more than one clearly defined tasks. This section should also be more concise than "Miksi tehdään?".
1. Only if the descriptions are empty: Use "title:" of the ticket to create the standardized description. If both "title:" and "description:" are empty, write "Sisältö puuttuu." to both.
2. Descriptions which dont clearly enough answer to all the topics "Miksi tehdään?" and "Tehtävät", here is a breakdown how to fill them with standardized description.
    a. Miksi tehdään?: Tehtävän tarvetta tai kuvausta ei ole luotu.
    b. Tehtävät: Toteutustapaa ei ole vahvistettu.
3. The original descriptions are in finnish, please keep your standardized ticket descriptions in finnish.
4. Read both the "title:" and "description:" to understand the contents of the ticket better. You can use the content in "title:" in order to write better standardized descriptions.
5. If you evaluate that the description is grammatically incorrect, please write grammatically correct standardized description outputs.
6. Output **ONLY** a JSON array. No numbering, no bullets, no markdown, no explanations. Keep the original "id" key from input_data and use the "id" in the output.
Example of item no 6:
[
  {"id": 1, "description": "First standardized text"},
  {"id": 2, "description": "Second standardized text"}
]

Example Jira ticket Title and Description:
title: "Nuppilukudatan haku yhtenäistetystä excelistä"
description: "Tiinalta tuli uus excel, jonka Ruotsi on ottanut käyttöön nuppiluvun raportoinnissa. Tässä on yhdistettynä HiQ, public ja lamia. Mukana myös aloittaneet ja lähteneet. Vaihdetaan powerbi:n queryt viittaamaan tähän yhteen yhtenäiseen exceliin."

Example Assistant(you) response:
description: "Miksi tehdään?: Tarve siirtää nuppiluvun raportointia uuteen Excel-tiedostoon, joka on Ruotsin käytössä. Tehtävät: Vaihdetaan Power BI:n kyselyt viittaamaan Tiinan uuteen Excel-tiedostoon, jossa on yhdistettynä HiQ, public ja lamia sekä myös aloittaneet ja lähteneet henkilöt.
"""

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 19, Finished, Available, Finished)

### Limit tokens and call API
Get Azure token and items from Azure Key Vault. The OpenAI API key is stored in the key vault and this is what we want to fetch.

In [13]:
# how many per batch?
batch_size = 5
batches = [
    input_data[i : i + batch_size]
    for i in range(0, 50, batch_size)
]

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 20, Finished, Available, Finished)

In [14]:
# Feed the OpenAI model and append outputs to a list.

standardized_descriptions = []
requests_sent = 0
start_minute = time.time()

for batch in batches:
    # Throttle to max 25 requests/minute
    requests_sent += 1
    elapsed = time.time() - start_minute
    if requests_sent >= 25:
        # if we've sent 25 requests in under 60 seconds, sleep out the remainder
        if elapsed < 60:
            time.sleep(60 - elapsed)
        # reset
        start_minute = time.time()
        requests_sent = 0

    batch_content = json.dumps(batch, ensure_ascii=False)
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user",   "content": batch_content},
        ],
        max_tokens=1500,       # cap per-call; adjust if needed
        temperature=1.0,
        top_p=1.0,
    )
    # print(response)

    out = json.loads(response.choices[0].message.content)
    standardized_descriptions.extend(out)

    # Optional: small sleep to smooth out token rate over minute
    time.sleep(0.5)  # ~120 sec paced → 120 requests/hour


StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 21, Finished, Available, Finished)

### Create dataframes
Create dataframe from the list and join the new data to the original dataframe.

In [ ]:
# Create df from function output and rename columns
output_df = spark.createDataFrame(standardized_descriptions)
output_df = output_df.withColumn("new_description", col("description"))
output_df = output_df.select("id", "new_description")
# output_df.show(10)
# display(output_df)

In [ ]:
# Join old df with new df
df_joined = flat_issues.join(output_df, on="id", how="left")

# TEST
# df_joined.show(10)
# df_filt = df_joined.filter(df_joined["id"] == 41)
# df_filt.show()

In [23]:
# BUILDING FINAL DATAFRAME

final_df = df_joined.select(
    "id",
    "title",
    "description", 
    "new_description", 
    "assignee",
    "comments",
    "priority",
    "status",
    "created",
    "updated"
)

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 30, Finished, Available, Finished)

### Write the dataframe to delta table

In [26]:
# Write the delta table
final_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/system_of_work/" + tableName)

StatementMeta(, bd12a972-daa3-4c81-bcf2-bd79081ad353, 33, Finished, Available, Finished)